# DLB report: usage example

This notebook creates a report from the NetCDF fixtures in `tests/test_files`. Run it with the `wetb` package available to the selected Jupyter kernel.

The generated files are written to `tutorial_report` so the existing example report is left untouched.

In [ ]:
from pathlib import Path

from wetb.dlb.report_generator.dlb_components import Component
from wetb.dlb.report_generator.dlb_report import DLBReport

# This works when the notebook is opened in dlb/report_generator or from the repository root.
REPORT_DIR = Path.cwd()
if not (REPORT_DIR / 'tests' / 'test_files').is_dir():
    REPORT_DIR = Path.cwd() / 'dlb' / 'report_generator'

TEST_FILES_DIR = REPORT_DIR / 'tests' / 'test_files'
OUTPUT_DIR = REPORT_DIR / 'tutorial_report'
if not TEST_FILES_DIR.is_dir():
    raise FileNotFoundError(f'Could not find test files at {TEST_FILES_DIR}')

print(f'Input data: {TEST_FILES_DIR}')
print(f'Report output: {OUTPUT_DIR}')

## Configure the report

`REPORT_CONFIG` defines input/output locations and report-wide DLC settings. Each entry in `COMPONENTS` selects sensors by a name or description match and chooses the report sections to generate.

Available sections are `statistics`, `extreme`, `equivalent`, `directional_extreme`, and `directional_equivalent`. Use `sections='all'` to generate every available section.

Set `output_formats` to `['latex']`, `['markdown']`, or `['latex', 'markdown']`. LaTeX remains the default when `output_formats` is omitted.

In [ ]:
REPORT_CONFIG = {
    'output_dir': OUTPUT_DIR,
    'cases': {'baseline': TEST_FILES_DIR},
    # Choose ['latex'], ['markdown'], or both.
    'output_formats': ['latex', 'markdown'],
    'main_tex_filename': 'main.tex',
    'markdown_filename': 'report.md',
    'dlcs': 'all',  # Or, for example: ['12', '24', '31']
    'fatigue_dlc_list': ['12', '24', '31', '41', '64'],
    'report_title': 'DLB tutorial report',
}

COMPONENTS = (
    Component('Tower Base', sections='all', fatigue_m=4, match='Tower Base'),
    Component('Blade 1 root', sections=('extreme',), fatigue_m=10,
              match='Blade 1 root'),
)

## Generate the report

The call returns the `DLBReport` instance, which exposes the resolved components, loaded case data, and generated configuration for interactive inspection.

In [ ]:
report = DLBReport(components=COMPONENTS, **REPORT_CONFIG).generate()

print('Components:', list(report.components))
print('Cases:', list(report.case_data))

## Inspect the output

Compile `main.tex` with your usual LaTeX toolchain to produce the PDF when LaTeX output is enabled. Markdown output embeds generated figures and renders the generated tables where possible; each table also retains a link to its LaTeX source. The cell below lists generated report sources and previews the enabled report documents.

In [ ]:
tex_files = sorted(path.relative_to(OUTPUT_DIR) for path in OUTPUT_DIR.rglob('*.tex'))
for path in tex_files:
    print(path)

if 'latex' in REPORT_CONFIG.get('output_formats', ['latex']):
    main_tex = OUTPUT_DIR / REPORT_CONFIG.get('main_tex_filename', 'main.tex')
    print('\n--- LaTeX preview ---')
    print(main_tex.read_text(encoding='utf-8')[:1000])

if 'markdown' in REPORT_CONFIG.get('output_formats', []):
    markdown_report = OUTPUT_DIR / REPORT_CONFIG.get('markdown_filename', 'report.md')
    print('\n--- Markdown preview ---')
    print(markdown_report.read_text(encoding='utf-8')[:1000])

## Compare two cases

To generate comparison plots, replace the `cases` entry with two named post-processing folders. Both cases must contain corresponding sensor IDs.

```python
REPORT_CONFIG['cases'] = {
    'baseline': Path('path/to/baseline/input_files'),
    'updated': Path('path/to/updated/input_files'),
}
report = DLBReport(components=COMPONENTS, **REPORT_CONFIG).generate()
```